In [ ]:
import kagglehub
import os
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

In [ ]:
# Task 2: Write your code here:
print(f"Dataset shape: {df_food.shape}")
df_food.head()

In [ ]:
# Task 3: Write your code here:
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
#to check target distribution
df_food['Delivery_Time'].hist()

In [ ]:
# Task 1: Write your code here:
df_food.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
#Check for missing values
print("Missing values:")
df_food.isnull().sum()

In [ ]:
df_clean = df_food.copy()

categorical_cols = df_clean.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('unknown')


df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df_food['Delivery_Time'].mean()) # fill target with mean
df_clean = df_clean.dropna(subset=['Courier_Experience_yrs'])


In [ ]:
print("Missing values:")
df_clean.isnull().sum()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df_clean):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
#One hot encoding
# Encode categorical columns - converts text to integers
from sklearn.preprocessing import LabelEncoder


print('data before encoding:\n', categorical_cols) #show before encoding
for col in categorical_cols: #instead of doing it to each column alone
   label_encoder = LabelEncoder()
   df_clean[col] = label_encoder.fit_transform(df_clean[col].astype(str)) #some values are floats so we convert to string so it can be transformed properly


print('data before encoding:\n', categorical_cols) #show before encoding


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
#there's high frequency between the time 35 and 50

In [ ]:
# Task 1: Write your code here:
#Order_ID	Distance_km	Weather	Traffic_Level	Time_of_Day	Vehicle_Type	Preparation_Time_min	Courier_Experience_yrs
# Define features (X) and target (y)
feature_cols = ['Order_ID','Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs']
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from sklearn.metrics import mean_absolute_error


kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/5")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Random Forest Regressor
  random_forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
    )
  random_forest_model.fit(X_train, y_train)

  # Predict
  y_pred = random_forest_model.predict(X_test)

  # Calculate metrics
  mae=mean_absolute_error(y_test, y_pred)

  print(f"\n{random_forest_model}:")
  print(f"  MAE:  {np.mean(random_forest_model[mae]):.4f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
import matplotlib.pyplot as plt

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': random_forest_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
df_food['Delivery_Time'].hist()

In [ ]:
# Task Bonus: Write your code here: